# MP18 — CNN Edge Detection Laboratory

Compare fixed Sobel derivatives with a learned convolution trained to imitate them. This is supervised operator imitation, not independent semantic-edge detection or evidence that learning beats Sobel. Learner mastery remains unassessed.

## Intuition, shapes and units
An edge is a local change in intensity. Gx responds to change across columns; Gy responds to change across rows. A vertical boundary therefore produces a horizontal derivative. We use single-channel NCHW float32 inputs, with two output channels for signed Gx and Gy. The fixed 3×3 kernels are divided by eight. Padding replicates boundary pixels; zero padding is compared separately.

Normalized Gx is [[-1,0,1],[-2,0,2],[-1,0,1]]/8 and Gy is its transpose. Magnitude M=sqrt(Gx²+Gy²) is nonnegative and loses sign. These are intensity-change units per pixel step, not calibrated physical gradients. We use cross-correlation, not reversed mathematical convolution.

## Hand example
In a 3×3 vertical step patch [[0,0,1],[0,0,1],[0,0,1]], Gx=(1+2+1)/8=0.5 and Gy=0, so magnitude=0.5. For a constant patch, both derivative sums vanish. At a zero-padded image boundary, a constant positive image suddenly meets artificial zeros and produces a false edge.

In [ ]:
from pathlib import Path
import json, torch
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
gx=torch.tensor([[-1.,0,1],[-2,0,2],[-1,0,1]])/8
patch=torch.tensor([[0.,0,1],[0,0,1],[0,0,1]])
assert float((patch*gx).sum())==.5
assert float((patch*gx.T).sum())==0
assert float(gx.sum())==0
print('Vertical step: Gx=0.5, Gy=0, magnitude=0.5')

## Training specification and separation
Run `python scripts/edge_experiment.py`. The training fixtures are 32 seeded uniform-noise images, each 16×16. Their two-channel targets come from the declared fixed Sobel operator. Noise excites all coefficients, avoiding the ambiguity of training on only one straight edge. A randomly initialized Conv2d(1,2,3,bias=False), with 18 trainable coefficients, minimizes mean squared component error. Adam uses learning rate 0.03, full batch 32, 500 steps, seed 18, CPU and two threads. No scheduler or data augmentation is used. The final fixed-step model is inspected; validation does not choose a checkpoint.

Validation fixtures are a vertical step, horizontal step and diagonal line, distinct from the noise training inputs. The existing real NAIP training crop is an additional demonstration, not held-out research evaluation. No test scene or new download is used. Weights are regenerated deterministically by the script and are not committed.

Loss is mean((predicted_G - target_G)²) across batch, two components and all pixels. Comparing signed components prevents magnitude-only ambiguities: opposite signed derivatives have the same magnitude.

In [ ]:
report=json.loads((ROOT/'reports/mp18-edges.json').read_text())
assert report['validation_component_mse']<1e-10
assert report['max_kernel_error']<1e-4
assert report['replicate_constant_max']==0
assert report['zero_constant_max']==.5
print(json.dumps(report,indent=2))
from IPython.display import Image, display
display(Image(filename=str(ROOT/'reports/mp18-edges.png')))

## What actually happened
The learned kernels match the known kernels to approximately 8.68e-9 maximum coefficient error. Synthetic validation component MSE is approximately 1.44e-17. This is expected because the target generator is exactly representable by the model; it is not a discovery of a better edge detector.

The real example is a checksummed 64×64 NAIP red-band training crop from ROI_00006, acquired 2021-09-17, public domain with USDA/OpenSR attribution. Stored values divided by 10000 are numerical scaling, not physical reflectance calibration. Fixed and learned magnitude panels share a display range. Their near equality shows transfer of the learned linear operator to this input, not accuracy against human boundary annotations.

Replicate padding produces zero derivatives on a constant image. Zero padding produces a maximum absolute component of 0.5 at its boundary. This artifact can dominate a visualization; always document padding. Sobel also responds to noise, texture and shadows, which may not be meaningful object boundaries.

## Practice — solve first
1. Why does a vertical boundary activate Gx rather than Gy?
2. Compute the magnitude for Gx=0.3,Gy=0.4.
3. How many trainable coefficients does this model have? What changes if bias is enabled?
4. Could this experiment show that learned edges beat Sobel? Explain.
5. Why is signed-component loss more informative than magnitude-only loss?

## Worked solutions
1. The intensity changes as column position changes, which is the x direction.
2. sqrt(0.09+0.16)=0.5.
3. Two output channels times one input channel times 3×3 equals 18. Two biases would increase it to 20.
4. No. Sobel generated every target. Independent annotated boundaries and a prespecified evaluation protocol would be needed for semantic accuracy claims.
5. Reversing both signs leaves magnitude unchanged. Component loss detects that error; magnitude-only loss cannot uniquely identify the derivative orientation.

## Reflection
Learned features depend on the objective and supervision. When labels are a known filter, training can recover that filter. Semantic edge detection is a different problem requiring independent labels. Keep this distinction when interpreting later CNN feature maps.

References: [project brief](README.md) and [MP17 convolution](../../notebooks/MP17.ipynb).